# **PopOut — Estratégias de Pesquisa Adversarial e Árvores de Decisão**
### Inteligência Artificial 2025/2026

| | |
|---|---|
| **Grupo** | TP 6 Grupo 6 |
| **Elementos** | Eduardo Moura — nº202406710 · Filipe Huang — nº202406540 · Diego Nóbrega — nº202407575 |
---

## Índice
1. [Introdução](#1-introdução)
2. [O Jogo PopOut](#2-o-jogo-popout)
3. [Monte Carlo Tree Search (MCTS)](#3-monte-carlo-tree-search-mcts)
4. [Geração do Dataset](#4-geração-do-dataset)
5. [Árvore de Decisão — ID3](#5-árvore-de-decisão--id3)
6. [Computador vs. Computador](#6-computador-vs-computador)
7. [Resultados e Discussão](#7-resultados-e-discussão)
8. [Conclusão](#8-conclusão)


---
## **1. Introdução**

Este trabalho tem como objetivo implementar e avaliar dois agentes de Inteligência Artificial capazes de jogar **PopOut**, uma variante do *Connect-4*:

- **MCTS** (*Monte Carlo Tree Search*) — algoritmo de pesquisa adversarial que estima a qualidade de cada jogada através de simulações aleatórias.
- **ID3** (*Iterative Dichotomiser 3*) — algoritmo de aprendizagem supervisionada que constrói uma árvore de decisão a partir de um dataset gerado pelo MCTS.

### Restrições consideradas
- Não foram utilizadas bibliotecas de aprendizagem automática (e.g., `scikit-learn`) para treinar ou definir as árvores de decisão.
- O dataset de treino do ID3 foi gerado internamente através de simulações MCTS.
- Toda a lógica de jogo, pesquisa e aprendizagem foi implementada de raiz.

---
## **2. O Jogo PopOut**

O **PopOut** é uma variante do *Connect-4* jogado num tabuleiro de 6 linhas × 7 colunas.
Em cada turno, o jogador pode:
- **Put** — inserir uma peça própria no topo de uma coluna;
- **Pop** — remover uma peça própria da linha de fundo da mesma coluna (todas as peças acima descem uma posição).

O objetivo é alinhar 4 peças consecutivas na horizontal, vertical ou diagonal.

### Regras especiais
1. **Vitória simultânea** (após *pop*): se ambos ficarem com 4 em linha, vence quem fez o *pop*.
2. **Tabuleiro cheio**: o jogador em turno pode declarar empate em vez de jogar.
3. **Repetição de estados**: se o mesmo estado ocorrer 3 vezes, qualquer jogador pode declarar empate.

### Representação do estado
→ Implementado em `game/game.py` — classe `PopOutState`

| Atributo | Descrição |
|---|---|
| `board` | Matriz 6×7 — `0` (vazio), `1` (X), `2` (O) |
| `player` | Jogador atual (`1` ou `2`) |
| `history` | Dicionário de estados para a regra de repetição |
| `last_move_pop` | Booleano para a regra de vitória simultânea |
| `draw_state` | Booleano para estdo de empate |


In [ ]:
# game.py – classe PopOutState
class PopOutState:
    
    def __init__(self):
        self.board = [[0]*7 for _ in range(6)]
        self.player = 1
        self.history = {}
        self.last_move_pop = False
        self.update_history()
        self.draw_state = False
    
    def copy(self):
        new = PopOutState()
        new.board = [row[:] for row in self.board]
        new.player = self.player
        new.history = dict(self.history)
        new.last_move_pop = self.last_move_pop
        new.draw_state = self.draw_state
        return new
    
    def state_hash(self):
        return tuple(tuple(row) for row in self.board)
    
    def update_history(self):
        h = self.state_hash()
        self.history[h] = self.history.get(h, 0) + 1
    
    def get_legal_moves(self):
        moves = []
        for c in range(7):
            if any(self.board[r][c]==0 for r in range(6)):
                moves.append((c, 'put'))
            if self.board[0][c] == self.player:
                moves.append((c, 'pop'))
        if self.is_full():
            moves.append((-1, 'draw'))
        return moves
    
    def make_move(self, moves):
        col, action = moves
        new_state = self.copy()
        new_state.last_move_pop = False
        if action == 'draw':
            new_state.draw_state = True
            new_state.player = 3 - self.player
            new_state.update_history()
            return new_state
        if action == 'pop':
            for r in range(5):
                new_state.board[r][col] = new_state.board[r+1][col]
            new_state.board[5][col] = 0
            new_state.last_move_pop = True
        else:  # put
            for r in range(6):
                if new_state.board[r][col] == 0:
                    new_state.board[r][col] = new_state.player
                    break
        new_state.player = 3 - self.player
        new_state.update_history()
        return new_state
    
    def has_four_in_row(self, player):
        board = self.board
        dirs = [(0,1),(1,0),(1,1),(1,-1)]
        for r in range(6):
            for c in range(7):
                if board[r][c] != player: continue
                for dr, dc in dirs:
                    if 0 <= r+3*dr < 6 and 0 <= c+3*dc < 7:
                        if (board[r][c]==player and
                            board[r+dr][c+dc]==player and
                            board[r+2*dr][c+2*dc]==player and
                            board[r+3*dr][c+3*dc]==player):
                            return True
        return False
    
    def is_full(self):
        return all(all(cell!=0 for cell in row) for row in self.board)

    def get_winner(self):
        if self.draw_state:
            return 'draw'
        if any(count>=3 for count in self.history.values()):
            return 'draw'
        p1 = self.has_four_in_row(1)
        p2 = self.has_four_in_row(2)
        if p1 and p2:
            return 3 - self.player if self.last_move_pop else None
        if p1: return 1
        if p2: return 2
        return None
    
    def is_terminal(self):
        return self.get_winner() is not None

### Interface de jogo
→ Implementada em `game/interface.py` — função `play_game()`

Suporta três modos:
1. **Humano vs. Humano**
2. **Humano vs. Computador** (MCTS ou ID3)
3. **Computador vs. Computador** (MCTS vs. ID3)


In [ ]:
def print_board(state):
    print("\n      0   1   2   3   4   5   6")
    print("   " + "─" * 29)
    for row in state.board[::-1]:
        line = " │ ".join(["X" if x == 1 else "O" if x == 2 else "-" for x in row])
        print("   │ " + line + " │")
    print("   " + "─" * 29)
    if not state.is_terminal(): print(f"   It is now {'X' if state.player == 1 else 'O'}'s turn.\n")

def play_game():
    while True:
        state = PopOutState()
        
        print("\n" + "=" * 35)
        print("           POPOUT GAME")
        print("=" * 35)
        print("Rules: Drop or Pop your pieces.\nFirst to 4 in a row wins!")
        print("Type 'put' or 'pop' when asked.\n")

        ai_mcts_player = None
        ai_id3_player = None

        while True:
            print("\nMain Menu:")
            print("(1) Human vs Human")
            print("(2) Human vs AI")
            print("(3) AI (MCTS) vs AI (ID3)")
            print("(4) Exit")
            mode = input("Choose mode (1/2/3/4): ").strip()
            if mode in ['1', '2', '3', '4']:
                break
            print("Please enter 1, 2, 3, or 4.")

        if mode == '4':
            print("Exiting game. Goodbye!")
            break

        if mode == '2':
            while True:
                print("\nChoose AI opponent:")
                print("(1) MCTS")
                print("(2) ID3")
                ai_choice = input("Choice (1/2): ").strip()
                if ai_choice in ['1', '2']:
                    break
                print("Please enter 1 or 2.")

            while True:
                symbol = input("Do you want to play as 'X' (first) or 'O' (second)? (X/O): ").strip().upper()
                if symbol in ['X', 'O']:
                    break
                print("Please enter X or O.")
                
            if symbol == 'X':
                if ai_choice == '1': ai_mcts_player = 2
                else: ai_id3_player = 2
            else:
                if ai_choice == '1': ai_mcts_player = 1
                else: ai_id3_player = 1

        elif mode == '3':
            while True:
                first = input("Who plays first as 'X'? (1) MCTS or (2) ID3: ").strip()
                if first in ['1', '2']:
                    break
                print("Please enter 1 or 2.")
                
            if first == '1':
                ai_mcts_player = 1
                ai_id3_player = 2
            else:
                ai_id3_player = 1
                ai_mcts_player = 2

        tree, features = None, None
        if ai_id3_player is not None:
            print("\nLoading dataset and training ID3 Tree... Please wait.")
            tree, features = train_tree(max_depth=10)
            if not tree:
                print("Warning: dataset.csv not found. ID3 will play randomly.")

        print_board(state)

        while not state.is_terminal():
            player_symbol = "X" if state.player == 1 else "O"
            print(f"\n{'=' * 35}")
            print(f"            {player_symbol}'S TURN")
            print(f"{'=' * 35}")

            moves = state.get_legal_moves()

            if ai_mcts_player and state.player == ai_mcts_player:
                print("MCTS AI is thinking...")
                move = mcts_search(state, iterations=1000)
                print(f"MCTS played: column {move[0]}, {move[1]}")
                state = state.make_move(move)
                print_board(state)
                continue

            if ai_id3_player and state.player == ai_id3_player:
                print("ID3 AI is thinking...")
                if tree:
                    move = get_id3_move(state, tree, features)
                    if move not in moves:
                        move = moves[0]
                else:
                    import random
                    move = random.choice(moves)
                
                print(f"ID3 played: column {move[0]}, {move[1]}")
                state = state.make_move(move)
                print_board(state)
                continue

            print(f"Legal moves: {moves}")

            is_draw = any(m[1]=='draw' for m in moves)

            if is_draw: text = "\nEnter column (0-6) or 'd' for DRAW: "
            else: text = "\nEnter column (0-6): "

            while True:
                try:
                    choice = input(text)
                    if choice=='d':
                        if is_draw:
                            state = state.make_move((-1, 'draw'))
                            break
                        else: continue
                    col = int(choice)
                    if not 0 <= col <= 6:
                        print("Column must be between 0 and 6!")
                        continue

                    can_put = any(state.board[r][col] == 0 for r in range(6))
                    can_pop = (state.board[0][col] == state.player)

                    if not can_put and not can_pop:
                        print("No moves possible in this column.")
                        continue
                    if can_put and can_pop:
                        ask = input("Enter action ('+' for put or '-' for pop): ").strip().lower()
                        if ask not in ['+', '-']:
                            print("Please type '+' or '-'.")
                            continue
                        action = 'pop' if ask == '-' else 'put'
                    elif can_put:
                        print(f"Only 'put' is possible in column {col}.")
                        action = 'put'
                    else:
                        print(f"Only 'pop' is possible in column {col}.")
                        action = 'pop'
                    
                    move = (col, action)
                    if move in moves:
                        state = state.make_move(move)
                        break
                    else: print("This move is not legal.")
                except ValueError: print("Please enter a valid number for column.")
            
            print_board(state)

        #Game Over
        print("\n" + "=" * 35)
        winner = state.get_winner()
        if winner == "draw": print("             GAME DRAW!")
        elif winner == 1: print("             'X' WINS!")
        elif winner == 2: print("             'O' WINS!")
        print("=" * 35)

---
## **3. Monte Carlo Tree Search (MCTS)**

O MCTS é um algoritmo de **pesquisa adversarial** que constrói iterativamente uma árvore
orientada por simulações aleatórias (*rollouts*). Cada iteração passa por 4 fases:

| Fase | Descrição |
|---|---|
| **1. Seleção** | Percorre a árvore usando UCT até encontrar um nó não totalmente expandido |
| **2. Expansão** | Adiciona um novo nó filho para uma jogada ainda não explorada |
| **3. Simulação** | Executa um *rollout* aleatório até ao estado terminal |
| **4. Retropropagação** | Propaga o resultado para todos os nós ancestrais |

### Critério de seleção — UCT (*Upper Confidence Bound for Trees*)

$$UCT(i) = \frac{w_i}{n_i} + c \cdot \sqrt{\frac{\ln N}{n_i}}$$

Onde $w_i$ = vitórias, $n_i$ = visitas ao nó $i$, $N$ = visitas ao pai, $c$ = constante de exploração ($\sqrt{2} \approx 1.41$).

→ Implementado em `game/mcts.py` — classe `MCTSNode`, função `mcts_search()`


In [ ]:
class MCTSNode:
    def __init__(self, state, parent=None, move=None):
        self.state = state
        self.parent = parent
        self.move = move
        self.children = []
        self.wins = 0
        self.visits = 0
        self.untried_moves = state.get_legal_moves()

    def is_fully_expanded(self):
        return len(self.untried_moves) == 0

    def best_child(self, c=1.41):
        return max(self.children, key=lambda n:
            n.wins / n.visits + c * math.sqrt(math.log(self.visits) / n.visits)
        )

    def expand(self):
        move = self.untried_moves.pop()
        new_state = self.state.make_move(move)
        child = MCTSNode(new_state, parent=self, move=move)
        self.children.append(child)
        return child

    def rollout(self):
        state = self.state.copy()
        while not state.is_terminal():
            moves = state.get_legal_moves()
            state = state.make_move(random.choice(moves))
        return state.get_winner()

    def backpropagate(self, result, ai_player):
        self.visits += 1
        if result == 'draw':
            self.wins -= 0.5
        elif result == ai_player:
            self.wins += 1
        else: self.wins -= 15
        if self.parent:
            self.parent.backpropagate(result, ai_player)


def mcts_search(state, iterations=1500, c=1.41):
    total_pieces = sum(cell != 0 for row in state.board for cell in row)
    if total_pieces < 2:
        moves = state.get_legal_moves()
        return random.choice(moves)

    ai_player = state.player
    root = MCTSNode(state)

    for _ in range(iterations):
        node = root
        while node.is_fully_expanded() and node.children:
            node = node.best_child(c)

        if not node.is_fully_expanded():
            node = node.expand()

        result = node.rollout()

        node.backpropagate(result, ai_player)

    best = max(root.children, key=lambda n: n.visits)
    return best.move

### Variantes exploradas

| Variante | Parâmetro alterado | Objetivo |
|---|---|---|
| **MCTS Padrão** | `iterations=1000`, `c=1.41` | Baseline de referência |
| **Exploração agressiva** | `c=2.0`, `c=2.5` | Favorece nós pouco visitados |
| **Exploração conservadora** | `c=0.5`, `c=0.8` | Favorece nós com bom histórico |

**Início aleatório**: Primeiras 2 peças aleatórias (Aumenta variedade do jogo)

Os resultados comparativos estão na Secção 7.


---
## 4. Geração do Dataset

Para treinar a árvore ID3, foi gerado um dataset de pares $\langle estado_i, jogada_i \rangle$
através de simulações de jogos MCTS vs. MCTS.

### Codificação das features
→ Implementado em `game/generate_dataset.py` — função `state_to_features()`

| Features | Descrição | Valores |
|---|---|---|
| `cell_0_0` a `cell_5_6` | 42 células do tabuleiro | `0`, `1`, `2` |
| `player` | Jogador atual | `1` ou `2` |
| `move` (label) | Melhor jogada (MCTS) | ex: `"3_put"`, `"2_pop"` |

### Parâmetros de geração
| Parâmetro | Valor | Justificação |
|---|---|---|
| Jogos simulados | 1000 | Equilíbrio entre diversidade e tempo |
| Iterações MCTS/jogada | 2000 | Labels de alta qualidade |
| Deduplicação | Sim | Evita overfitting em estados repetidos |

→ Dataset em `game/dataset.csv`


In [ ]:
import csv
from game import PopOutState
from mcts import mcts_search

def state_to_features(state):
    flat = [cell for row in state.board for cell in row]
    flat.append(state.player)
    return flat

def generate_dataset(num_games=1000, iterations=2000, output_file="dataset.csv"):
    header = [f"cell_{r}_{c}" for r in range(6) for c in range(7)] + ["player", "move"]
    total = 0
    seen_states = set()

    with open(output_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(header)

        for game_num in range(num_games):
            state = PopOutState()
            print(f"Game {game_num + 1}/{num_games} | Exemplos guardados: {total}")

            while not state.is_terminal():
                best_move = mcts_search(state, iterations=iterations, c=1.41)

                features = state_to_features(state)
                state_key = tuple(features)

                if state_key not in seen_states:
                    seen_states.add(state_key)
                    col, action = best_move
                    label = f"{col}_{action}"
                    writer.writerow(features + [label])
                    f.flush()
                    total += 1

                state = state.make_move(best_move)

    print(f"\nDataset gerado com sucesso!")
    print(f"Total de exemplos únicos: {total}")
    print(f"Ficheiro guardado em: '{output_file}'")

if __name__ == "__main__":
    generate_dataset(num_games=1000, iterations=2000, output_file="dataset.csv")

---
## 5. Árvore de Decisão — ID3

O **ID3** (*Iterative Dichotomiser 3*) constrói uma árvore de decisão recursivamente,
selecionando em cada nó o atributo com maior **Ganho de Informação**.

**Entropia** de um conjunto $S$:
$$H(S) = -\sum_{c \in C} p_c \log_2 p_c$$

**Ganho de Informação** para o atributo $A$:
$$IG(S, A) = H(S)-\sum_{v\in valores(A)} \frac{|S_v|}{|S|} \cdot H(S_v)$$

> **Nota:** Sem uso de `scikit-learn` ou equivalentes. Implementação de raiz.

### 5.1 Base ID3

Base code that is the same for both datesets.


In [ ]:
import math
from collections import Counter

class Node:
    def __init__(self, feature=None, label=None):
        self.feature = feature
        self.label = label
        self.children = {}
        self.majority_class = None

    def is_leaf(self):
        return self.label is not None

def entropy(data, label):
    n = len(data)
    if n == 0:
        return 0
    counts = Counter(row[label] for row in data)
    return -sum((c / n) * math.log2(c / n) for c in counts.values() if c > 0)

def information_gain(data, feature, label):
    n = len(data)
    base_entropy = entropy(data, label)
    values = set(row[feature] for row in data)
    weighted = sum(
        (len(subset := [r for r in data if r[feature] == v]) / n) * entropy(subset, label)
        for v in values
    )
    return base_entropy - weighted

def id3(data, features, label, depth=0, max_depth=10):
    classes = [row[label] for row in data]
    majority = Counter(classes).most_common(1)[0][0]

    if len(set(classes)) == 1:
        return Node(label=classes[0])

    if not features or depth >= max_depth:
        return Node(label=majority)

    best_feat = max(features, key=lambda f: information_gain(data, f, label))
    node = Node(feature=best_feat)
    node.majority_class = majority

    values = set(row[best_feat] for row in data)
    remaining = [f for f in features if f != best_feat]

    for val in values:
        subset = [row for row in data if row[best_feat] == val]
        if not subset:
            node.children[val] = Node(label=majority)
        else:
            node.children[val] = id3(subset, remaining, label, depth + 1, max_depth)

    return node

def predict(node, row):
    if node.is_leaf():
        return node.label
    val = row.get(node.feature)
    if val not in node.children:
        if node.majority_class is not None:
            return node.majority_class
        return predict(list(node.children.values())[0], row)
    return predict(node.children[val], row)

def print_tree(node, indent=0, branch=""):
    prefix = "    " * indent
    if branch:
        print(prefix + f"[{branch}]")
    if node.is_leaf():
        print(prefix + f"  - CLASS: {node.label}")
    else:
        print(prefix + f"  SPLIT ON: {node.feature}")
        for val, child in sorted(node.children.items()):
            print_tree(child, indent + 1, val)

### 5.2 Warm-up: Dataset Iris

Validámos a implementação no **Iris** antes de a aplicar ao PopOut.
Como o Iris tem atributos numéricos contínuos, foi implementada **discretização** por *bins* iguais.

→ Implementado em `game/id3_iris.py`

In [ ]:
import csv
import random

def load_iris(filepath="iris.csv"):
    data = []
    with open(filepath, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            data.append({
                "sepallength": float(row["sepallength"]),
                "sepalwidth":  float(row["sepalwidth"]),
                "petallength": float(row["petallength"]),
                "petalwidth":  float(row["petalwidth"]),
                "class":       row["class"]
            })
    return data

def discretize(data, features, n_bins=3):
    bin_labels = ["low", "mid", "high"] if n_bins == 3 else [f"b{i}" for i in range(n_bins)]
    thresholds = {}
    for feat in features:
        values = [row[feat] for row in data]
        min_v, max_v = min(values), max(values)
        step = (max_v - min_v) / n_bins
        thresholds[feat] = [min_v + step * i for i in range(1, n_bins)]
    disc_data = []
    for row in data:
        new_row = dict(row)
        for feat in features:
            bin_idx = sum(row[feat] > t for t in thresholds[feat])
            new_row[feat] = bin_labels[bin_idx]
        disc_data.append(new_row)
    return disc_data, thresholds, bin_labels

def discretize_row(row, features, thresholds, bin_labels):
    new_row = dict(row)
    for feat in features:
        bin_idx = sum(row[feat] > t for t in thresholds[feat])
        new_row[feat] = bin_labels[bin_idx]
    return new_row

def train_test_split(data, test_ratio=0.2, seed=42):
    random.seed(seed)
    shuffled = data[:]
    random.shuffle(shuffled)
    split = int(len(shuffled) * (1 - test_ratio))
    return shuffled[:split], shuffled[split:]

def evaluate(tree, test_data, thresholds, bin_labels, features):
    correct = 0
    for row in test_data:
        disc = discretize_row(row, features, thresholds, bin_labels)
        pred = predict(tree, disc)
        if pred == row["class"]:
            correct += 1
    return correct / len(test_data)

if __name__ == "__main__":
    FEATURES = ["sepallength", "sepalwidth", "petallength", "petalwidth"]

    data = load_iris("iris.csv")
    print(f"Exemplos carregados: {len(data)}")

    train, test = train_test_split(data, test_ratio=0.2)
    print(f"Treino: {len(train)} | Teste: {len(test)}")

    disc_train, thresholds, bin_labels = discretize(train, FEATURES, n_bins=3)
    tree = id3(disc_train, FEATURES, label="class", max_depth=10)

    print("\n=== ÁRVORE DE DECISÃO (IRIS) ===")
    print_tree(tree)

    acc = evaluate(tree, test, thresholds, bin_labels, FEATURES)
    print(f"\nPrecisão no teste: {acc * 100:.1f}%")

### 5.3 ID3 Aplicado ao PopOut

O mesmo algoritmo ID3 é treinado sobre o dataset do PopOut.
As features do tabuleiro já são discretas (`0`, `1`, `2`), sem necessidade de discretização adicional.

→ Implementado em `game/id3_popout.py`
- `train_id3_popout(filepath)` — carrega o dataset, treina a árvore ID3
- `id3_move(state, tree)` — dado um estado, devolve a jogada prevista


In [ ]:
import csv

def load_dataset(filepath="dataset.csv"):
    data = []
    with open(filepath, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            processed = {}
            for k, v in row.items():
                if k == "move":
                    processed[k] = v
                else:
                    processed[k] = int(v)
            data.append(processed)
    return data

def train_tree(max_depth=12, filepath="dataset.csv"):
    try:
        data = load_dataset(filepath)
        features = [f for f in data[0].keys() if f != "move"]
        tree = id3(data, features, label="move", max_depth=max_depth)
        return tree, features
    except FileNotFoundError:
        print(f"Erro: ficheiro '{filepath}' não encontrado.")
        return None, None

def get_id3_move(state, tree, features):
    state_dict = {}
    for r in range(6):
        for c in range(7):
            state_dict[f"cell_{r}_{c}"] = state.board[r][c]
    state_dict["player"] = state.player

    prediction = predict(tree, state_dict)
    if prediction is None:
        return None
    col_str, action = prediction.split("_")
    return (int(col_str), action)

Exemplo de utilização (descomentar para testar):
if __name__ == "__main__":
    tree, features = train_tree(max_depth=12, filepath="dataset.csv")
    if tree:
        print("Árvore treinada com sucesso!")
        print("Primeiros níveis da árvore:")
        print_tree(tree, max_depth=3)
    else:
        print("Não foi possível treinar a árvore.")

---
## 6. Computador vs. Computador

Para comparar os dois agentes, foram realizados jogos automáticos MCTS vs. ID3.

→ Lógica de torneio em `game/interface.py` ou `game/tournament.py`


In [ ]:
# Torneio automático: MCTS vs. ID3
# Descomente após o ID3 estar integrado

# def torneio(agente_x, agente_o, n_jogos=50):
#     resultados = {'X (MCTS)': 0, 'O (ID3)': 0, 'Empate': 0}
#     for i in range(n_jogos):
#         state = PopOutState()
#         while not state.is_terminal():
#             move = agente_x(state) if state.player == 1 else agente_o(state)
#             state = state.make_move(move)
#         w = state.get_winner()
#         if w == 1:    resultados['X (MCTS)'] += 1
#         elif w == 2:  resultados['O (ID3)'] += 1
#         else:         resultados['Empate'] += 1
#         print(f"  Jogo {i+1}/{n_jogos} — vencedor: {w}", end='
')
#     return resultados

# resultados = torneio(
#     agente_x=lambda s: mcts_search(s, iterations=1000),
#     agente_o=lambda s: id3_move(s, tree_popout),
#     n_jogos=50
# )
# print("\nResultados:", resultados)

print("⚠️  Descomente após integrar o agente ID3")


In [ ]:
# Visualização dos resultados do torneio
# Preenche com os valores reais após o torneio

# resultados = {'X (MCTS)': 28, 'O (ID3)': 16, 'Empate': 6}  # substituir pelos reais

# import matplotlib.pyplot as plt
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# axes[0].bar(resultados.keys(), resultados.values(),
#             color=['#4e8cff', '#ff6b6b', '#aaa'], edgecolor='white', width=0.5)
# axes[0].set_title('MCTS (X) vs. ID3 (O) — 50 jogos')
# axes[0].set_ylabel('Número de vitórias')
# for i, (k, v) in enumerate(resultados.items()):
#     axes[0].text(i, v + 0.3, str(v), ha='center', fontweight='bold')

# axes[1].pie(resultados.values(), labels=resultados.keys(), autopct='%1.1f%%',
#             colors=['#4e8cff', '#ff6b6b', '#aaa'])
# axes[1].set_title('Distribuição de resultados')

# plt.tight_layout()
# plt.show()

print("⚠️  Descomente após obter os resultados do torneio")


---
## 7. Resultados e Discussão

### 7.1 Resumo do desempenho

| Métrica | MCTS (1000 iter) | MCTS (3000 iter) | ID3 (PopOut) |
|---|---|---|---|
| Taxa de vitória vs. aleatório | X% | X% | X% |
| Taxa de vitória (MCTS vs. ID3) | X% | — | X% |
| Tempo médio por jogada | ~Xms | ~Xms | ~Xms |
| Qualidade da jogada | Alta | Muito alta | Depende do treino |
| Interpretabilidade | Baixa | Baixa | Alta |

*Preencher com valores reais após os experimentos.*

### 7.2 Efeito da constante de exploração $c$ no MCTS

| $c$ | Comportamento | Resultado observado |
|---|---|---|
| `0.5` | Mais exploração (nós com bom histórico) | [preencher] |
| `1.41` ($\sqrt{2}$, padrão) | Equilíbrio teórico exploração/exploração | [preencher] |
| `2.0` | Mais exploração (nós pouco visitados) | [preencher] |

### 7.3 Qualidade do ID3

- **Precisão no Iris**: X%
- **Precisão no PopOut**: X%
- **Observações sobre o dataset**: [descrever distribuição, balanceamento, cobertura de estados]
- **Limitações**: O ID3 só conhece estados vistos durante a geração — estados raros podem gerar jogadas sub-ótimas.


---
## 8. Conclusão

Neste trabalho foram implementados com sucesso:
- O jogo **PopOut** com todas as regras, incluindo as 3 regras especiais
- Um agente **MCTS** com critério UCT, com análise do impacto do número de iterações e da constante $c$
- Uma árvore de decisão **ID3** de raiz, validada no Iris e aplicada ao PopOut
- Um **dataset** de X exemplos únicos gerado por auto-jogo MCTS (2000 iterações/jogada)
- Modo **Computador vs. Computador** para comparação direta dos dois agentes

### Reflexão comparativa

O **MCTS** é mais forte porque avalia o estado em tempo real.
O **ID3** é mais rápido na inferência mas depende totalmente da qualidade e cobertura do dataset de treino.

### Possíveis melhorias
- Aumentar o dataset (mais jogos, mais variações)
- Usar profundidade máxima adaptativa no ID3
- Implementar *alpha-beta pruning* como terceiro agente para comparação
